# Liquid checkpoint conversion — `LiquidRecurrent` / `LiquidCell` fusion

The classes `LiquidRecurrent` and `LiquidCell` were merged into a single `LiquidCell`
(`arenai_agent/src/networks/recurrent/liquid_cell.h`). The intermediate `cell` submodule
disappeared, so every parameter saved under `liquid.cell.*` now lives under `liquid.*`:

| old key | new key |
|---|---|
| `liquid.cell.a` | `liquid.a` |
| `liquid.cell.raw_tau` | `liquid.raw_tau` |
| `liquid.cell.f.*` | `liquid.f.*` |
| `liquid.to_output.*` | unchanged |

This notebook reads an `actor.pt` saved **before** the fusion (via
`torch::serialize::OutputArchive`), remaps the keys and writes a checkpoint loadable by the
fused C++ code (`torch::serialize::InputArchive`). The same mapping applies to `critic.pt`.

Notes:
- the parameter **order** is unchanged by the fusion, so `actor_optim.pt` / `critic_optim.pt`
  need no conversion;
- requires `torch` (`pip install torch`, the CPU wheel is enough — it is not part of the
  `arenai_agent/python` requirements).

In [ ]:
import re
from pathlib import Path

import torch
from torch import nn

OLD_CHECKPOINT = Path(
    "/home/samuel/CLionProjects/ArenAI/outputs/train_400_ppo_liquid_beta_silu_truncation-no-penalty/save_97/actor.pt"
)
# written under a `converted/` sibling folder, same file name
NEW_CHECKPOINT = OLD_CHECKPOINT.parent / "converted" / OLD_CHECKPOINT.name

## Read the old state dict

A module saved by LibTorch's `OutputArchive` is a TorchScript archive: `torch.jit.load`
gives back the full module hierarchy with its named parameters.

In [ ]:
old_module = torch.jit.load(str(OLD_CHECKPOINT), map_location="cpu")

old_params = {name: p.detach().clone() for name, p in old_module.named_parameters()}
old_buffers = dict(old_module.named_buffers())
assert not old_buffers, f"unexpected buffers, extend the notebook: {sorted(old_buffers)}"

for name, p in old_params.items():
    print(f"{name:45s} {tuple(p.shape)}")

## Remap and rebuild

The C++ `Module::load` walks the archive by **module hierarchy**, and requires every
serialized submodule to exist — including parameterless ones (SiLU, Sigmoid, LayerNorm
positions in the `Sequential`s...). So the converted archive must replicate the full module
tree, not just the parameter paths: the old `liquid.cell` node is dropped and its children
are re-attached directly to `liquid`.

In [ ]:
def convert(name: str) -> str:
    return re.sub(r"^liquid\.cell\.", "liquid.", name)


class Node(nn.Module):
    def forward(self) -> None:
        return None


def ensure_path(root: Node, dotted: str) -> Node:
    module = root
    for part in dotted.split("."):
        if part not in module._modules:
            module.add_module(part, Node())
        module = module._modules[part]
    return module


root = Node()
for name, _ in old_module.named_modules():
    if name and name != "liquid.cell":
        ensure_path(root, convert(name))
for name, tensor in old_params.items():
    *path, leaf = convert(name).split(".")
    ensure_path(root, ".".join(path)).register_parameter(leaf, nn.Parameter(tensor))

NEW_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
torch.jit.save(torch.jit.script(root), str(NEW_CHECKPOINT))
print(f"written: {NEW_CHECKPOINT}")

## Verify the round trip

In [ ]:
reloaded = torch.jit.load(str(NEW_CHECKPOINT), map_location="cpu")

new_params = dict(reloaded.named_parameters())
expected = {convert(name): t for name, t in old_params.items()}

assert set(new_params) == set(expected), set(new_params) ^ set(expected)
for name, t in expected.items():
    assert torch.equal(new_params[name], t), name

# structure the C++ InputArchive will walk
liquid = reloaded.liquid
for attr in ("a", "raw_tau", "f", "to_output"):
    assert hasattr(liquid, attr), attr
assert not hasattr(liquid, "cell")

print(f"OK — {len(new_params)} parameters remapped")
for name in sorted(new_params):
    if name.startswith("liquid"):
        print(f"  {name:35s} {tuple(new_params[name].shape)}")